# 06 — Design Your Algorithm

You have exact (02), the measurement loop (03), fuzzy (04), and blocking (05). Now we **design** the full pipeline: **combining** exact and fuzzy, and a **blended** score so you have one threshold to tune. Same measurement loop—evaluate and compare. We use the evaluation set (50 rows, 30 known pairs) with `full_name` for fuzzy.

**Back:** [05 Blocking](05_blocking.ipynb) · **Next:** [07 Deduplication](07_deduplication.ipynb)

## 1. Load data and ground truth

We need the evaluation tables with `full_name` for fuzzy, and ground truth. **Data:** Same **evaluation set** as 02 and 03 (50×50, 30 perfect pairs) so combo and blended results are comparable to earlier notebooks.

In [ ]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher, SimpleEvaluator

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = Path.cwd() if (Path.cwd() / "tutorial_data").exists() else Path.cwd().parent.parent / "docs" / "tutorial"
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_evaluation

left, right, ground_truth = load_evaluation()
left = left.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
right = right.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Left: {left.shape[0]} rows, Right: {right.shape[0]} rows, Ground truth: {ground_truth.shape[0]} pairs")

## 2. The improvement loop

This is the loop you'll use from here on: (1) run matcher with your current rules and blocking, (2) evaluate against ground truth, (3) change one thing (rules, blocking key, fuzzy threshold, or blend cutoff), (4) re-run and compare. Repeat until precision and recall are where you need them. No magic—just measure, tune, compare.

## 3. Exact + fuzzy combo

Often you want both: exact matches (e.g. email) *and* fuzzy matches (e.g. name) that catch pairs where email is missing or different. Run exact and fuzzy separately, merge the pair sets, dedupe by (id, id_right), then evaluate the combined set. You'll see whether the combo improves recall without hurting precision too much.

In [ ]:
exact_results = matcher.match(rules="email")
fuzzy_results = matcher.match_fuzzy(field="full_name", threshold=0.85)

exact_pairs = exact_results.matches.select(["id", "id_right"]).unique()
fuzzy_pairs = fuzzy_results.matches.select(["id", "id_right"]).unique()
combo_pairs = pl.concat([exact_pairs, fuzzy_pairs]).unique(subset=["id", "id_right"])

metrics_exact = exact_results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
metrics_combo = SimpleEvaluator().evaluate(combo_pairs, ground_truth, left_id_col="id", right_id_col="id_right")
print("Exact only: ", f"precision={metrics_exact['precision']:.2%}, recall={metrics_exact['recall']:.2%}")
print("Exact+fuzzy:", f"precision={metrics_combo['precision']:.2%}, recall={metrics_combo['recall']:.2%}")

## 4. Blended algorithm: one score per pair

Instead of "exact OR fuzzy" as separate outputs, you can **blend** them: assign a score to each source (e.g. exact = 1.0, fuzzy = confidence), group by pair and take the max (or another aggregation), then apply a single cutoff. One score per pair, one threshold to tune—simpler to reason about and to ship.

In [ ]:
exact = matcher.match(rules=["email", ["first_name", "last_name"]])
exact_df = exact.matches.select(["id", "id_right"]).unique().with_columns(pl.lit(1.0).alias("score"))

fuzzy_name = matcher.match_fuzzy(field="full_name", threshold=0.5)
fuzzy_df = fuzzy_name.matches.select(["id", "id_right", "confidence"]).with_columns(pl.col("confidence").alias("score"))

all_pairs = pl.concat([
    exact_df.select(["id", "id_right", "score"]),
    fuzzy_df.select(["id", "id_right", "score"])
])
blended = all_pairs.group_by(["id", "id_right"]).agg(pl.col("score").max().alias("blended_score"))
final = blended.filter(pl.col("blended_score") >= 0.85)

metrics = SimpleEvaluator().evaluate(final, ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Blended (max score, cutoff 0.85): precision={metrics['precision']:.2%}, recall={metrics['recall']:.2%}, F1={metrics['f1']:.2%}")

You can try different cutoffs (e.g. 0.82 vs 0.85) or a weighted blend instead of max; the loop is the same—evaluate on the same ground truth and compare.

**Next:** [07 Deduplication](07_deduplication.ipynb) — same ideas on a single table.